In [5]:
# import pandas as pd
# from pathlib import Path
# import matplotlib.pyplot as plt
# import pandas as pd
# import numpy as np
# BASE_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/RawFile")

# DAYS = [
#     "19-01-2026",
#     "20-01-2026",
#     "21-01-2026",
#     "22-01-2026",
#     "23-01-2026",
# ]

# rows = []   # ← list of DataFrames (to be concatenated)

# def process_minimpl(MiniMPL: pd.DataFrame) -> pd.DataFrame:
#     nrb_max = MiniMPL["copol_nrb"].max()
#     norm = MiniMPL["copol_nrb"] / nrb_max if nrb_max not in (0, None) else 0

#     return pd.DataFrame({
#         "range_raw": MiniMPL["range_raw"],
#         "range_m_for_NRB": MiniMPL["range_raw"] * 1000,
#         "range_m": MiniMPL["range_nrb"] * 1000,
#         "copol_raw": MiniMPL["copol_raw"],
#         "copol_snr": MiniMPL["copol_snr"],
#         "copol_nrb": MiniMPL["copol_nrb"],
#         "pbls": MiniMPL["pbls"],
#         "Normalize_copol_nrb": norm,
#     })


# for day in DAYS:
#     day_folder = BASE_DIR / f"{day}-DATfile"
#     if not day_folder.exists():
#         print(f"❌ Folder missing: {day_folder}")
#         continue

#     date_obj = pd.to_datetime(day, format="%d-%m-%Y")
#     date_str = date_obj.strftime("%Y%m%d")

#     times = pd.date_range(
#         start=f"{date_str} 00:00",
#         end=f"{date_str} 23:55",
#         freq="5min"
#     )

#     print(f"\n📂 Processing {day_folder}")

#     missing_count = 0
#     processed_count = 0

#     for t in times:
#         ts_str = t.strftime("%Y%m%d%H%M")
#         infile = day_folder / f"MPL_5038_{ts_str}.csv"

#         if not infile.exists():
#             missing_count += 1
#             continue

#         try:
#             MiniMPL = pd.read_csv(infile)
#         except Exception as e:
#             print(f"⚠️ Read error: {infile.name} → {e}")
#             continue

#         df = process_minimpl(MiniMPL)

#         df.insert(0, "timestamp", t)
#         df.insert(0, "day", day)

#         rows.append(df)
#         processed_count += 1

#     print(
#         f"✅ {day}: processed {processed_count} files | "
#         f"❌ missing {missing_count} files"
#     )


# MiniMPL_ALL = pd.concat(rows, ignore_index=True)



📂 Processing /Users/thunthita/Lidarforiypnb/LIDar/RawFile/19-01-2026-DATfile
✅ 19-01-2026: processed 288 files | ❌ missing 0 files

📂 Processing /Users/thunthita/Lidarforiypnb/LIDar/RawFile/20-01-2026-DATfile
✅ 20-01-2026: processed 130 files | ❌ missing 158 files

📂 Processing /Users/thunthita/Lidarforiypnb/LIDar/RawFile/21-01-2026-DATfile
✅ 21-01-2026: processed 288 files | ❌ missing 0 files

📂 Processing /Users/thunthita/Lidarforiypnb/LIDar/RawFile/22-01-2026-DATfile
✅ 22-01-2026: processed 288 files | ❌ missing 0 files

📂 Processing /Users/thunthita/Lidarforiypnb/LIDar/RawFile/23-01-2026-DATfile
✅ 23-01-2026: processed 288 files | ❌ missing 0 files


In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

BASE_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/RawFile/")
OUT_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict/")
OUT_DIR.mkdir(exist_ok=True)

DAYS = [
    "26-01-2026",
    "27-01-2026",
    "28-01-2026",
    "29-01-2026",
    "30-01-2026",
]

def process_minimpl(MiniMPL: pd.DataFrame) -> pd.DataFrame:
    nrb_max = MiniMPL["copol_nrb"].max()
    if not np.isfinite(nrb_max) or nrb_max == 0:
        norm = np.nan
    else:
        norm = MiniMPL["copol_nrb"] / nrb_max

    return pd.DataFrame({
        "range_raw": MiniMPL["range_raw"],
        "range_m_for_NRB": MiniMPL["range_raw"] * 1000,
        "range_m": MiniMPL["range_nrb"] * 1000,
        "copol_raw": MiniMPL["copol_raw"],
        "copol_snr": MiniMPL["copol_snr"],
        "copol_nrb": MiniMPL["copol_nrb"],
        "pbls": MiniMPL["pbls"],
        "Normalize_copol_nrb": norm,
    })

# =========================
# Main loop (per day)
# =========================
for day in DAYS:
    day_folder = BASE_DIR / f"{day}-DATfile"
    if not day_folder.exists():
        print(f"❌ Folder missing: {day_folder}")
        continue

    date_obj = pd.to_datetime(day, format="%d-%m-%Y")
    date_str = date_obj.strftime("%Y%m%d")

    times = pd.date_range(
        start=f"{date_str} 00:00",
        end=f"{date_str} 23:55",
        freq="5min"
    )

    print(f"\n📂 Processing {day}")

    rows_day = []          # ← reset every day
    missing_count = 0
    processed_count = 0

    for t in times:
        ts_str = t.strftime("%Y%m%d%H%M")
        infile = day_folder / f"MPL_5038_{ts_str}.csv"

        if not infile.exists():
            missing_count += 1
            continue

        try:
            MiniMPL = pd.read_csv(infile)
        except Exception as e:
            print(f"⚠️ Read error: {infile.name} → {e}")
            continue

        df = process_minimpl(MiniMPL)
        df.insert(0, "timestamp", t)
        df.insert(0, "day", day)

        rows_day.append(df)
        processed_count += 1

    if rows_day:
        MiniMPL_day = pd.concat(rows_day, ignore_index=True)

        out_file = OUT_DIR / f"MiniMPL_{day}.csv"
        MiniMPL_day.to_csv(out_file, index=False)

        print(
            f"✅ {day}: saved {processed_count} files → {out_file.name} | "
            f"❌ missing {missing_count}"
        )
    else:
        print(f"⚠️ {day}: no valid data files")



📂 Processing 26-01-2026
✅ 26-01-2026: saved 288 files → MiniMPL_26-01-2026.csv | ❌ missing 0

📂 Processing 27-01-2026
✅ 27-01-2026: saved 108 files → MiniMPL_27-01-2026.csv | ❌ missing 180

📂 Processing 28-01-2026
✅ 28-01-2026: saved 287 files → MiniMPL_28-01-2026.csv | ❌ missing 1

📂 Processing 29-01-2026
✅ 29-01-2026: saved 287 files → MiniMPL_29-01-2026.csv | ❌ missing 1

📂 Processing 30-01-2026
✅ 30-01-2026: saved 287 files → MiniMPL_30-01-2026.csv | ❌ missing 1


In [2]:
# Where your per-day combined CSVs are
IN_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict/")

# Where you want the extracted files to go
OUT_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/")
OUT_DIR.mkdir(parents=True, exist_ok=True)


for day in DAYS:
    in_file = IN_DIR / f"MiniMPL_{day}.csv"
    if not in_file.exists():
        print(f"❌ Missing: {in_file}")
        continue

    df = pd.read_csv(in_file)

    # Make sure timestamp is datetime
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    # Keep only times at minute 05 and 35 (00:05, 00:35, ... 23:35)
    mask = df["timestamp"].dt.minute.isin([5, 35])

    # Keep needed columns
    df_sel = df.loc[mask, ["day", "timestamp", "pbls"]].copy()

    # Reduce to 1 row per timestamp (since pbls repeats across range bins)
    # Option A: take first value per timestamp
    df_out = df_sel.groupby(["day", "timestamp"], as_index=False).first()

    # Rename + create derived columns
    df_out = df_out.rename(columns={"pbls": "pbls_km"})
    df_out["pbls_m"] = df_out["pbls_km"] * 1000.0
    df_out["min_pbls_m"] = df_out["pbls_m"] - 300.0
    df_out["max_pbls_m"] = df_out["pbls_m"] + 300.0

    # Reorder columns nicely
    df_out = df_out[["day", "timestamp", "pbls_km", "pbls_m", "min_pbls_m", "max_pbls_m"]]

    out_file = OUT_DIR / f"pbls_{day}_0005_0035_to_2335.csv"
    df_out.to_csv(out_file, index=False)

    print(f"✅ Saved: {out_file} | rows={len(df_out)}")


✅ Saved: /Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/pbls_26-01-2026_0005_0035_to_2335.csv | rows=48
✅ Saved: /Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/pbls_27-01-2026_0005_0035_to_2335.csv | rows=18
✅ Saved: /Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/pbls_28-01-2026_0005_0035_to_2335.csv | rows=48
✅ Saved: /Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/pbls_29-01-2026_0005_0035_to_2335.csv | rows=47
✅ Saved: /Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/pbls_30-01-2026_0005_0035_to_2335.csv | rows=48


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

# Where your per-day combined CSVs are
IN_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict/")

# Where you want the extracted files to go
OUT_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Expected timestamps: every hour at :05 and :35
EXPECTED_TIMES = pd.to_timedelta(
    [f"{h:02d}:{m:02d}:00" for h in range(24) for m in (5, 35)]
)

for day in DAYS:
    in_file = IN_DIR / f"MiniMPL_{day}.csv"
    if not in_file.exists():
        print(f"❌ Missing: {in_file}")
        continue

    df = pd.read_csv(in_file)

    # Make sure timestamp is datetime
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    # Keep only minute 05 and 35
    mask = df["timestamp"].dt.minute.isin([5, 35])
    df_sel = df.loc[mask, ["day", "timestamp", "pbls"]].copy()

    # Reduce to 1 row per timestamp
    df_sel = (
        df_sel
        .groupby(["day", "timestamp"], as_index=False)
        .first()
    )

    # -------------------------------------------------
    # Build FULL expected timestamp table for that day
    # -------------------------------------------------
    day_ts = pd.Timestamp(day)
    full_index = pd.DataFrame({
        "day": day,
        "timestamp": day_ts + EXPECTED_TIMES
    })

    # -------------------------------------------------
    # Merge → missing timestamps become NaN
    # -------------------------------------------------
    df_out = full_index.merge(
        df_sel,
        on=["day", "timestamp"],
        how="left"
    )

    # Derived columns (NaN-safe)
    df_out = df_out.rename(columns={"pbls": "pbls_km"})
    df_out["pbls_m"] = df_out["pbls_km"] * 1000.0
    df_out["min_pbls_m"] = df_out["pbls_m"] - 300.0
    df_out["max_pbls_m"] = df_out["pbls_m"] + 300.0

    # Reorder columns
    df_out = df_out[
        ["day", "timestamp", "pbls_km", "pbls_m", "min_pbls_m", "max_pbls_m"]
    ]

    out_file = OUT_DIR / f"pbls_{day}_0005_0035_to_2335.csv"
    df_out.to_csv(out_file, index=False)

    print(f"✅ Saved: {out_file} | rows={len(df_out)}")


✅ Saved: /Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/pbls_26-01-2026_0005_0035_to_2335.csv | rows=48
✅ Saved: /Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/pbls_27-01-2026_0005_0035_to_2335.csv | rows=48
✅ Saved: /Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/pbls_28-01-2026_0005_0035_to_2335.csv | rows=48
✅ Saved: /Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/pbls_29-01-2026_0005_0035_to_2335.csv | rows=48
✅ Saved: /Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/pbls_30-01-2026_0005_0035_to_2335.csv | rows=48


In [6]:
# def plot_profile_at_timestamp(
#     df: pd.DataFrame,
#     ts,
#     *,
#     xcol: str,
#     ycol: str,
#     figsize=(6, 3),
#     xlabel=None,
#     ylabel=None,
#     title=None,
#     xscale="linear",
#     yscale="linear",
#     xlim=None,
#     ylim=None,
#     show=True
# ):
#     """
#     Plot ycol vs xcol for a single timestamp.

#     Parameters
#     ----------
#     df : pd.DataFrame
#         Long-format DataFrame (must contain 'timestamp')
#     ts : str or pd.Timestamp
#         Timestamp to plot
#     xcol, ycol : str
#         Column names for x and y axes
#     """

#     ts = pd.Timestamp(ts)
#     df_t = df[df["timestamp"] == ts]

#     if df_t.empty:
#         raise ValueError(f"No data found for timestamp {ts}")

#     if xcol not in df_t.columns or ycol not in df_t.columns:
#         raise KeyError(f"Columns '{xcol}' or '{ycol}' not found")

#     plt.figure(figsize=figsize)
#     plt.plot(df_t[xcol], df_t[ycol])

#     plt.xlabel(xlabel if xlabel else xcol)
#     plt.ylabel(ylabel if ylabel else ycol)

#     if title:
#         plt.title(title)
#     else:
#         plt.title(f"{ycol} vs {xcol}\n{ts}")

#     plt.grid(True, alpha=0.3)
#     plt.xscale(xscale)
#     plt.yscale(yscale)

#     if xlim:
#         plt.xlim(*xlim)
#     if ylim:
#         plt.ylim(*ylim)

#     plt.tight_layout()

#     if show:
#         plt.show()

# plot_profile_at_timestamp(
#     MiniMPL_ALL,
#     "2026-01-05 00:05",
#     xcol="range_m",
#     ycol="copol_raw",
#     xlabel="Range (m)",
#     ylabel="Copol raw"
# )

# plot_profile_at_timestamp(
#     MiniMPL_ALL,
#     "2026-01-05 00:05",
#     xcol="range_m",
#     ycol="copol_nrb",
#     xlabel="Range (m)",
#     ylabel="Copol nrb"
# )
# plot_profile_at_timestamp(
#     MiniMPL_ALL,
#     "2026-01-05 00:05",
#     xcol="range_m",
#     ycol="Normalize_copol_nrb",
#     xlabel="Range (m)",
#     ylabel="Normalize Copol nrb"
# )


In [7]:
MINIMPL_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict")
PROTO_BASE = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV")

for day in DAYS:
    minimipl_csv = MINIMPL_DIR / f"MiniMPL_{day}.csv"
    if not minimipl_csv.exists():
        print(f"❌ Missing MiniMPL CSV: {minimipl_csv.name}")
        continue

    # 👉 อ่าน MiniMPL ของวันนั้น
    MiniMPL_day = pd.read_csv(minimipl_csv, parse_dates=["timestamp"])
    MiniMPL_day["timestamp"] = MiniMPL_day["timestamp"].dt.floor("min")

    print(f"\n📅 Processing {day}")



📅 Processing 26-01-2026

📅 Processing 27-01-2026

📅 Processing 28-01-2026

📅 Processing 29-01-2026

📅 Processing 30-01-2026


In [8]:
MiniMPL_day

,day,timestamp,range_raw,range_m_for_NRB,range_m,copol_raw,copol_snr,copol_nrb,pbls,Normalize_copol_nrb
0,30-01-2026,2026-01-30 00:00:00,0.029979,29.979246,119.91698,24.604744,80011.810000,0.278124,2.098547,0.930887
1,30-01-2026,2026-01-30 00:00:00,0.059958,59.958490,149.89623,11.891105,38663.820000,0.288657,NaN,0.966139
2,30-01-2026,2026-01-30 00:00:00,0.089938,89.937740,179.87548,10.567916,34361.742000,0.288124,NaN,0.964357
3,30-01-2026,2026-01-30 00:00:00,0.119917,119.916980,209.85472,10.438926,33945.477000,0.282255,NaN,0.944712
4,30-01-2026,2026-01-30 00:00:00,0.149896,149.896230,239.83397,10.388205,33783.055000,0.281857,NaN,0.943379
...,...,...,...,...,...,...,...,...,...,...
286995,30-01-2026,2026-01-30 23:55:00,29.859330,29859.330000,NaN,0.000714,-0.001931,NaN,NaN,NaN
286996,30-01-2026,2026-01-30 23:55:00,29.889309,29889.309000,NaN,0.000580,-0.455544,NaN,NaN,NaN
286997,30-01-2026,2026-01-30 23:55:00,29.919287,29919.287000,NaN,0.000818,0.343158,NaN,NaN,NaN
286998,30-01-2026,2026-01-30 23:55:00,29.949266,29949.266000,NaN,0.000625,-0.276386,NaN,NaN,NaN


In [9]:
def plot_profile_at_timestamp(
    df: pd.DataFrame,
    ts,
    *,
    xcol: str,
    ycol: str,
    figsize=(6, 3),
    xlabel=None,
    ylabel=None,
    title=None,
    xscale="linear",
    yscale="linear",
    xlim=None,
    ylim=None,
    show=False,
    save_path: Path | None = None,
    dpi=200,
    bbox_inches="tight",
):
    ts = pd.Timestamp(ts)
    df_t = df[df["timestamp"] == ts]

    if df_t.empty:
        return False  # reliable in loops

    if xcol not in df_t.columns or ycol not in df_t.columns:
        raise KeyError(f"Columns '{xcol}' or '{ycol}' not found")

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(df_t[xcol], df_t[ycol])

    ax.set_xlabel(xlabel if xlabel else xcol)
    ax.set_ylabel(ylabel if ylabel else ycol)

    if title:
        ax.set_title(title)
    else:
        ax.set_title(f"{ycol} vs {xcol}\n{ts}")

    ax.grid(True, alpha=0.3)
    ax.set_xscale(xscale)
    ax.set_yscale(yscale)

    if xlim:
        ax.set_xlim(*xlim)
    if ylim:
        ax.set_ylim(*ylim)

    fig.tight_layout()

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=dpi, bbox_inches=bbox_inches)

    if show:
        plt.show()

    plt.close(fig)
    return True


In [11]:
OUT_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict")

PLOTS = [
    dict(ycol="copol_raw",           ylabel="Copol raw"),
    dict(ycol="copol_nrb",           ylabel="Copol nrb"),
    dict(ycol="Normalize_copol_nrb", ylabel="Normalize Copol nrb"),
]

for day in DAYS:
    daily_csv = OUT_DIR / f"MiniMPL_{day}.csv"
    if not daily_csv.exists():
        print(f"❌ Missing daily CSV: {daily_csv.name}")
        continue

    df_day = pd.read_csv(daily_csv, parse_dates=["timestamp"])

    # ✅ make timestamp matching robust (กันกรณีมีวินาที/เศษ)
    df_day["timestamp"] = df_day["timestamp"].dt.floor("min")

    # โฟลเดอร์ output ต่อวัน
    day_out = OUT_DIR / day
    day_out.mkdir(parents=True, exist_ok=True)

    # ✅ สร้าง date_str ต่อวัน (แก้จุดพังของคุณ)
    date_obj = pd.to_datetime(day, format="%d-%m-%Y")
    date_str = date_obj.strftime("%Y-%m-%d")

    # ทุก 30 นาที เริ่มที่ 00:05
    times_30min = pd.date_range(
        start=f"{date_str} 00:05",
        end=f"{date_str} 23:55",
        freq="30min"
    )

    saved = 0
    missing_ts = 0

    for ts in times_30min:
        ts = pd.Timestamp(ts).floor("min")  # ✅ ให้ match แน่
        hhmm = ts.strftime("%H%M")

        # (optional) ถ้า timestamp นั้นไม่มีจริง ก็ข้ามเร็ว ๆ
        if not (df_day["timestamp"] == ts).any():
            missing_ts += 1
            continue

        for p in PLOTS:
            ycol = p["ycol"]
            ylabel = p["ylabel"]

            out_png = day_out / f"{ycol}_{hhmm}.png"

            ok = plot_profile_at_timestamp(
                df_day,
                ts,
                xcol="range_m",
                ycol=ycol,
                xlabel="Range (m)",
                ylabel=ylabel,
                xlim=(0, 5000),
                show=False,
                save_path=out_png,
            )
            if ok:
                saved += 1

    print(f"✅ {day}: Saved {saved} plots → {day_out} | missing timestamps: {missing_ts}")


✅ 20-01-2026: Saved 144 plots → /Users/thunthita/Lidarforiypnb/LIDar/src/Pict/20-01-2026 | missing timestamps: 0


In [12]:
# def plot_3profiles_at_timestamp(
#     df: pd.DataFrame,
#     ts,
#     *,
#     xcol="range_m",
#     ycols=("copol_raw", "copol_nrb", "Normalize_copol_nrb"),
#     labels=("Copol raw", "Copol nrb", "Normalize Copol nrb"),
#     xlim=None,
#     show=False,
#     save_path=None,
# ):
#     ts = pd.Timestamp(ts)
#     df_t = df[df["timestamp"] == ts]
#     if df_t.empty:
#         return False

#     plt.figure(figsize=(6, 3))
#     for y, lab in zip(ycols, labels):
#         plt.plot(df_t[xcol], df_t[y], label=lab)

#     plt.xlabel("Range (m)")
#     plt.ylabel("Signal")
#     plt.title(f"MiniMPL profiles\n{ts}")
#     plt.legend(loc="upper right")
#     plt.grid(True, alpha=0.3)

#     if xlim:
#         plt.xlim(*xlim)

#     plt.tight_layout()

#     if save_path:
#         plt.savefig(save_path, dpi=200, bbox_inches="tight")
#     if show:
#         plt.show()

#     plt.close()
#     return True


In [13]:
# OUT_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict")

# for day in DAYS:
#     daily_csv = OUT_DIR / f"MiniMPL_{day}.csv"
#     if not daily_csv.exists():
#         print(f"❌ Missing daily CSV: {daily_csv.name}")
#         continue

#     df_day = pd.read_csv(daily_csv, parse_dates=["timestamp"])
#     df_day["timestamp"] = df_day["timestamp"].dt.floor("min")  # ให้ match ชัวร์

#     day_out = OUT_DIR / day
#     day_out.mkdir(parents=True, exist_ok=True)

#     date_obj = pd.to_datetime(day, format="%d-%m-%Y")
#     date_str = date_obj.strftime("%Y-%m-%d")

#     times_30min = pd.date_range(
#         start=f"{date_str} 00:05",
#         end=f"{date_str} 23:55",
#         freq="30min"
#     )

#     saved = 0
#     for ts in times_30min:
#         ts = pd.Timestamp(ts).floor("min")
#         if not (df_day["timestamp"] == ts).any():
#             continue

#         hhmm = ts.strftime("%H%M")
#         out_png = day_out / f"MiniMPL_3profiles_{hhmm}.png"

#         ok = plot_3profiles_at_timestamp(
#             df_day,
#             ts,
#             xlim=(0, 5000),
#             save_path=out_png,
#             show=False
#         )
#         if ok:
#             saved += 1

#     print(f"✅ {day}: saved {saved} plots → {day_out}")


In [14]:
def plot_minimpl_vs_csv(
    MiniMPL_ALL: pd.DataFrame,
    *,
    ts,
    csv_path,
    xcol_minimpl,
    ycol_minimpl,
    xcol_csv,
    ycol_csv,
    label_minimpl="MiniMPL",
    label_csv="Prototype",
    figsize=(6, 3),
    xscale="linear",
    yscale="linear",
    xlim=None,
    ylim=None,
    title=None,
    show=True
):
    """
    Compare MiniMPL profile with external CSV profile at one timestamp.
    """

    ts = pd.Timestamp(ts)
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

    # ---- MiniMPL ----
    df_m = MiniMPL_ALL[MiniMPL_ALL["timestamp"] == ts]
    if df_m.empty:
        raise ValueError(f"No MiniMPL data for timestamp {ts}")

    # ---- External CSV ----
    df_c = pd.read_csv(csv_path)

    if xcol_csv not in df_c.columns or ycol_csv not in df_c.columns:
        raise KeyError("CSV columns not found")

    # ---- Plot ----
    plt.figure(figsize=figsize)

    plt.plot(
        df_m[xcol_minimpl],
        df_m[ycol_minimpl],
        label=label_minimpl,
        linewidth=2
    )

    plt.plot(
        df_c[xcol_csv],
        df_c[ycol_csv],
        label=label_csv,
        linestyle="--"
    )

    plt.xlabel(xcol_minimpl)
    plt.ylabel(ycol_minimpl)
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.xscale(xscale)
    plt.yscale(yscale)

    if xlim:
        plt.xlim(*xlim)
    if ylim:
        plt.ylim(*ylim)

    if title:
        plt.title(title)
    else:
        plt.title(f"{label_minimpl} vs {label_csv}\n{ts}")

    plt.tight_layout()

    if show:
        plt.show()

plot_minimpl_vs_csv(
    MiniMPL_ALL,
    ts="2026-01-19 00:05",
    csv_path="/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/19-01-2026-0005/19-01-2026-0005_processed.csv",
    xcol_minimpl="range_m",
    ycol_minimpl="Normalize_copol_nrb",
    xcol_csv="range_m",
    ycol_csv="range2_norm",
    label_minimpl="MiniMPL",
    label_csv="Prototype",
    yscale="linear",
    xlim=(0, 5000)
)


NameError: name 'MiniMPL_ALL' is not defined

In [ ]:
# plot_minimpl_vs_csv(
#     MiniMPL_ALL,
#     ts="2026-01-05 00:35",
#     csv_path="/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/05-01-2026-0035/05-01-2026-0035_processed.csv",
#     xcol_minimpl="range_m",
#     ycol_minimpl="Normalize_copol_nrb",
#     xcol_csv="range_m",
#     ycol_csv="range2_norm",
#     label_minimpl="MiniMPL",
#     label_csv="Prototype",
#     yscale="linear",
#     xlim=(0, 5000)
# )

In [ ]:
# plot_minimpl_vs_csv(
#     MiniMPL_ALL,
#     ts="2026-01-05 01:05",
#     csv_path="/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/05-01-2026-0105/05-01-2026-0105_processed.csv",
#     xcol_minimpl="range_m",
#     ycol_minimpl="Normalize_copol_nrb",
#     xcol_csv="range_m",
#     ycol_csv="range2_norm",
#     label_minimpl="MiniMPL",
#     label_csv="Prototype",
#     yscale="linear",
#     xlim=(0, 5000)
# )

In [ ]:
# def make_minimpl_prototype_plotter(
#     MiniMPL_ALL,
#     *,
#     day,
#     ts,
#     base_proto_dir
# ):
#     ts = pd.Timestamp(ts)
#     ts_str = ts.strftime("%H%M")

#     csv_path = (
#         Path(base_proto_dir)
#         / f"{day}-{ts_str}"
#         / f"{day}-{ts_str}_processed.csv"
#     )

#     def plot(**kwargs):
#         return plot_minimpl_vs_csv(
#             MiniMPL_ALL,
#             ts=ts,
#             csv_path=csv_path,
#             **kwargs
#         )

#     return plot


# plot_0105 = make_minimpl_prototype_plotter(
#     MiniMPL_ALL,
#     day="05-01-2026",
#     ts="2026-01-05 01:05",
#     base_proto_dir="/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV"
# )

# plot_0105(
#     xcol_minimpl="range_m",
#     ycol_minimpl="Normalize_copol_nrb",
#     xcol_csv="range_m",
#     ycol_csv="range2_norm",
#     label_minimpl="MiniMPL",
#     label_csv="Prototype",
#     xlim=(0, 5000)
# )


In [ ]:
# plot_1205 = make_minimpl_prototype_plotter(
#     MiniMPL_ALL,
#     day="05-01-2026",
#     ts="2026-01-05 12:05",
#     base_proto_dir="/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV"
# )

# plot_1205(
#     xcol_minimpl="range_m",
#     ycol_minimpl="Normalize_copol_nrb",
#     xcol_csv="range_m",
#     ycol_csv="range2_norm",
#     label_minimpl="MiniMPL",
#     label_csv="Prototype",
#     xlim=(0, 5000)
# )


In [ ]:
# plot_0505 = make_minimpl_prototype_plotter(
#     MiniMPL_ALL,
#     day="05-01-2026",
#     ts="2026-01-05 05:05",
#     base_proto_dir="/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV"
# )

# plot_0505(
#     xcol_minimpl="range_m",
#     ycol_minimpl="Normalize_copol_nrb",
#     xcol_csv="range_m",
#     ycol_csv="range2_norm",
#     label_minimpl="MiniMPL",
#     label_csv="Prototype",
#     xlim=(0, 5000)
# )


In [ ]:
# plot_0505 = make_minimpl_prototype_plotter(
#     MiniMPL_ALL,
#     day="05-01-2026",
#     ts="2026-01-05 05:35",
#     base_proto_dir="/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV"
# )

# plot_0505(
#     xcol_minimpl="range_m",
#     ycol_minimpl="Normalize_copol_nrb",
#     xcol_csv="range_m",
#     ycol_csv="range2_norm",
#     label_minimpl="MiniMPL",
#     label_csv="Prototype",
#     xlim=(0, 5000)
# )


In [26]:
def plot_minimpl_vs_csv(
    MiniMPL_ALL: pd.DataFrame,
    *,
    ts,
    csv_path,
    xcol_minimpl,
    ycol_minimpl,
    xcol_csv,
    ycol_csv,
    label_minimpl="MiniMPL",
    label_csv="Prototype",
    figsize=(6, 3),
    xscale="linear",
    yscale="linear",
    xlim=None,          # e.g. (0, 5000)
    ylim=None,
    title=None,
    show=True,
    find_peak=True,
    peak_xlim=(0, 5000),         # <-- focus peak search here
    annotate_peak=True,
):
    """
    Compare MiniMPL profile with external CSV profile at one timestamp,
    and (optionally) find peak positions within peak_xlim.
    Returns a dict with peak results when find_peak=True.
    """

    ts = pd.Timestamp(ts)
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

    # ---- MiniMPL ----
    df_m = MiniMPL_ALL[MiniMPL_ALL["timestamp"] == ts]
    if df_m.empty:
        raise ValueError(f"No MiniMPL data for timestamp {ts}")

    # ---- External CSV ----
    df_c = pd.read_csv(csv_path)

    if xcol_csv not in df_c.columns or ycol_csv not in df_c.columns:
        raise KeyError("CSV columns not found")

    # ---- Convert to numpy + clean NaNs ----
    xm = df_m[xcol_minimpl].to_numpy()
    ym = df_m[ycol_minimpl].to_numpy()
    xc = df_c[xcol_csv].to_numpy()
    yc = df_c[ycol_csv].to_numpy()

    # drop NaNs (important for argmax)
    m_ok = np.isfinite(xm) & np.isfinite(ym)
    c_ok = np.isfinite(xc) & np.isfinite(yc)
    xm, ym = xm[m_ok], ym[m_ok]
    xc, yc = xc[c_ok], yc[c_ok]

    results = None

    # ---- Find peaks within peak_xlim ----
    if find_peak and peak_xlim is not None:
        lo, hi = peak_xlim

        m_mask = (xm >= lo) & (xm <= hi)
        c_mask = (xc >= lo) & (xc <= hi)

        if not np.any(m_mask):
            raise ValueError(f"No MiniMPL data in peak_xlim={peak_xlim}")
        if not np.any(c_mask):
            raise ValueError(f"No CSV data in peak_xlim={peak_xlim}")

        # global max within window
        im = np.argmax(ym[m_mask])
        ic = np.argmax(yc[c_mask])

        peak_x_m = xm[m_mask][im]
        peak_y_m = ym[m_mask][im]
        peak_x_c = xc[c_mask][ic]
        peak_y_c = yc[c_mask][ic]

        results = {
            "ts": ts,
            "peak_xlim": peak_xlim,
            "minimpl_peak_x_m": float(peak_x_m),
            "minimpl_peak_y": float(peak_y_m),
            "csv_peak_x_m": float(peak_x_c),
            "csv_peak_y": float(peak_y_c),
            "peak_shift_m": float(peak_x_c - peak_x_m),  # csv - minimpl
        }

        print(f"[{ts}] Peak window: {lo}–{hi} m")
        print(f"  {label_minimpl} peak: x={peak_x_m:.1f} m, y={peak_y_m:.6g}")
        print(f"  {label_csv}     peak: x={peak_x_c:.1f} m, y={peak_y_c:.6g}")
        print(f"  Peak shift (csv - minimpl): {results['peak_shift_m']:.1f} m")

    # ---- Plot ----
    plt.figure(figsize=figsize)

    plt.plot(xm, ym, label=label_minimpl, linewidth=2)
    plt.plot(xc, yc, label=label_csv, linestyle="--")

    # draw peak markers
    if results and annotate_peak:
        plt.axvline(results["minimpl_peak_x_m"], linestyle=":", linewidth=1.5)
        plt.axvline(results["csv_peak_x_m"], linestyle=":", linewidth=1.5)
        plt.scatter([results["minimpl_peak_x_m"]], [results["minimpl_peak_y"]], s=30)
        plt.scatter([results["csv_peak_x_m"]], [results["csv_peak_y"]], s=30)

        plt.text(results["minimpl_peak_x_m"], results["minimpl_peak_y"],
                 f" {label_minimpl} peak\n {results['minimpl_peak_x_m']:.0f} m",
                 fontsize=8, va="top", ha="left")
        plt.text(results["csv_peak_x_m"], results["csv_peak_y"],
                 f" {label_csv} peak\n {results['csv_peak_x_m']:.0f} m",
                 fontsize=8, va="top", ha="right")

    plt.xlabel(xcol_minimpl)
    plt.ylabel(ycol_minimpl)
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.xscale(xscale)
    plt.yscale(yscale)

    # use xlim for viewing
    if xlim:
        plt.xlim(*xlim)
    if ylim:
        plt.ylim(*ylim)

    if title:
        plt.title(title)
    else:
        plt.title(f"{label_minimpl} vs {label_csv}\n{ts}")

    plt.tight_layout()

    if show:
        plt.show()

    return results

res = plot_minimpl_vs_csv(
    MiniMPL_ALL,
    ts="2026-01-19 00:05",
    csv_path="/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/19-01-2026-0005/19-01-2026-0005_processed.csv",
    xcol_minimpl="range_m",
    ycol_minimpl="Normalize_copol_nrb",
    xcol_csv="range_m",
    ycol_csv="range2_norm",
    label_minimpl="MiniMPL",
    label_csv="Prototype",
    yscale="linear",
    xlim=(0, 5000),          # plot view
    peak_xlim=(0, 5000),     # peak search window
)
print(res)


NameError: name 'MiniMPL_ALL' is not defined

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from pathlib import Path

# def plot_minimpl_vs_csv(
#     MiniMPL_ALL: pd.DataFrame,
#     *,
#     ts,
#     csv_path,
#     xcol_minimpl,
#     ycol_minimpl,
#     xcol_csv,
#     ycol_csv,
#     label_minimpl="MiniMPL",
#     label_csv="Prototype",
#     figsize=(6, 3),
#     xscale="linear",
#     yscale="linear",
#     xlim=None,          # e.g. (0, 5000)
#     ylim=None,
#     title=None,
#     show=True,
#     find_peak=True,
#     peak_xlim=(0, 5000),         # <-- focus peak search here
#     annotate_peak=False,         # <-- default: no text near peaks
#     peak_in_legend=False,        # <-- put peak values inside legend labels
#     peak_box=True,               # <-- put peak summary in a box (top-right)
#     legend_loc="upper right",
# ):
#     """
#     Compare MiniMPL profile with external CSV profile at one timestamp,
#     and (optionally) find peak positions within peak_xlim.
#     Returns a dict with peak results when find_peak=True.
#     """

#     ts = pd.Timestamp(ts)
#     csv_path = Path(csv_path)

#     if not csv_path.exists():
#         raise FileNotFoundError(csv_path)

#     # ---- MiniMPL ----
#     df_m = MiniMPL_ALL[MiniMPL_ALL["timestamp"] == ts]
#     if df_m.empty:
#         raise ValueError(f"No MiniMPL data for timestamp {ts}")

#     # ---- External CSV ----
#     df_c = pd.read_csv(csv_path)
#     if xcol_csv not in df_c.columns or ycol_csv not in df_c.columns:
#         raise KeyError("CSV columns not found")

#     # ---- Convert to numpy + clean NaNs ----
#     xm = df_m[xcol_minimpl].to_numpy()
#     ym = df_m[ycol_minimpl].to_numpy()
#     xc = df_c[xcol_csv].to_numpy()
#     yc = df_c[ycol_csv].to_numpy()

#     m_ok = np.isfinite(xm) & np.isfinite(ym)
#     c_ok = np.isfinite(xc) & np.isfinite(yc)
#     xm, ym = xm[m_ok], ym[m_ok]
#     xc, yc = xc[c_ok], yc[c_ok]

#     results = None

#     # ---- Find peaks within peak_xlim ----
#     if find_peak and peak_xlim is not None:
#         lo, hi = peak_xlim
#         m_mask = (xm >= lo) & (xm <= hi)
#         c_mask = (xc >= lo) & (xc <= hi)

#         if not np.any(m_mask):
#             raise ValueError(f"No MiniMPL data in peak_xlim={peak_xlim}")
#         if not np.any(c_mask):
#             raise ValueError(f"No CSV data in peak_xlim={peak_xlim}")

#         im = np.argmax(ym[m_mask])
#         ic = np.argmax(yc[c_mask])

#         peak_x_m = xm[m_mask][im]
#         peak_y_m = ym[m_mask][im]
#         peak_x_c = xc[c_mask][ic]
#         peak_y_c = yc[c_mask][ic]

#         results = {
#             "ts": ts,
#             "peak_xlim": peak_xlim,
#             "minimpl_peak_x_m": float(peak_x_m),
#             "minimpl_peak_y": float(peak_y_m),
#             "csv_peak_x_m": float(peak_x_c),
#             "csv_peak_y": float(peak_y_c),
#             "peak_shift_m": float(peak_x_c - peak_x_m),  # csv - minimpl
#         }

#         print(f"[{ts}] Peak window: {lo}–{hi} m")
#         print(f"  {label_minimpl} peak: x={peak_x_m:.1f} m, y={peak_y_m:.6g}")
#         print(f"  {label_csv}     peak: x={peak_x_c:.1f} m, y={peak_y_c:.6g}")
#         print(f"  Peak shift (csv - minimpl): {results['peak_shift_m']:.1f} m")

#     # ---- Plot ----
#     fig, ax = plt.subplots(figsize=figsize)

#     # labels (optionally include peak info)
#     lab_m = label_minimpl
#     lab_c = label_csv
#     if results and peak_in_legend:
#         lab_m = f"{label_minimpl} (peak {results['minimpl_peak_x_m']:.0f} m)"
#         lab_c = f"{label_csv} (peak {results['csv_peak_x_m']:.0f} m)"

#     ax.plot(xm, ym, label=lab_m, linewidth=2)
#     ax.plot(xc, yc, label=lab_c, linestyle="--")

#     # draw peak markers (no text by default)
#     if results:
#         ax.axvline(results["minimpl_peak_x_m"], linestyle=":", linewidth=1.5)
#         ax.axvline(results["csv_peak_x_m"], linestyle=":", linewidth=1.5)
#         ax.scatter([results["minimpl_peak_x_m"]], [results["minimpl_peak_y"]], s=30)
#         ax.scatter([results["csv_peak_x_m"]], [results["csv_peak_y"]], s=30)

#         if annotate_peak:
#             ax.text(
#                 results["minimpl_peak_x_m"], results["minimpl_peak_y"],
#                 f"{label_minimpl}\n{results['minimpl_peak_x_m']:.0f} m",
#                 fontsize=8, va="bottom", ha="left"
#             )
#             ax.text(
#                 results["csv_peak_x_m"], results["csv_peak_y"],
#                 f"{label_csv}\n{results['csv_peak_x_m']:.0f} m",
#                 fontsize=8, va="bottom", ha="right"
#             )

#         # peak summary box (top-right) -> ไม่ทับกราฟ
#         if peak_box:
#             box_text = (
#                 f"Peak (within {results['peak_xlim'][0]}–{results['peak_xlim'][1]} m)\n"
#                 f"{label_minimpl}: {results['minimpl_peak_x_m']:.0f} m\n"
#                 f"{label_csv}: {results['csv_peak_x_m']:.0f} m\n"
#                 f"Shift (csv - minimpl): {results['peak_shift_m']:.0f} m"
#             )
#             ax.text(
#             1.05, 0.98, box_text,        # อยู่นอกกราฟด้านขวา
#             transform=ax.transAxes,
#             ha="left", va="top",
#             fontsize=8,
#             bbox=dict(
#                 boxstyle="round,pad=0.3",
#                 facecolor="none",        # <<< ไม่มีสี
#                 edgecolor="0.4",         # สีกรอบเทาอ่อน
#                 linewidth=0.8
#             )
#         )


#     ax.set_xlabel(xcol_minimpl)
#     ax.set_ylabel(ycol_minimpl)
#     ax.grid(True, alpha=0.3)
#     ax.legend(loc=legend_loc)

#     ax.set_xscale(xscale)
#     ax.set_yscale(yscale)

#     if xlim:
#         ax.set_xlim(*xlim)
#     if ylim:
#         ax.set_ylim(*ylim)

#     if title:
#         ax.set_title(title)
#     else:
#         ax.set_title(f"{label_minimpl} vs {label_csv}\n{ts}")

#     fig.tight_layout()
#     if show:
#         plt.show()

#     return results

# res = plot_minimpl_vs_csv(
#     MiniMPL_ALL,
#     ts="2026-01-05 00:05",
#     csv_path="/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/05-01-2026-0005/05-01-2026-0005_processed.csv",
#     xcol_minimpl="range_m",
#     ycol_minimpl="Normalize_copol_nrb",
#     xcol_csv="range_m",
#     ycol_csv="range2_norm",
#     label_minimpl="MiniMPL",
#     label_csv="Prototype",
#     yscale="linear",
#     xlim=(0, 5000),
#     peak_xlim=(0, 5000),
#     annotate_peak=False,     # กันทับ
#     peak_box=True,           # กล่องมุมขวาบน
#     peak_in_legend=False,    # จะใส่เพิ่มใน legend ก็เปิดได้
#     legend_loc="upper right"
# )
# print(res)


In [28]:
def plot_minimpl_vs_csv(
    MiniMPL_ALL: pd.DataFrame,
    *,
    ts,
    csv_path,
    xcol_minimpl,
    ycol_minimpl,
    xcol_csv,
    ycol_csv,
    label_minimpl="MiniMPL",
    label_csv="Prototype",
    figsize=(6, 3),
    xscale="linear",
    yscale="linear",
    xlim=None,
    ylim=None,
    title=None,
    show=True,
    save_path=None,          # <<< ADD
    find_peak=True,
    peak_xlim=(0, 5000),
    annotate_peak=False,
    peak_in_legend=False,
    peak_box=True,
    legend_loc="upper right",
):

    ts = pd.Timestamp(ts)
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

    # ---- MiniMPL ----
    df_m = MiniMPL_ALL[MiniMPL_ALL["timestamp"] == ts]
    if df_m.empty:
        raise ValueError(f"No MiniMPL data for timestamp {ts}")

    # ---- External CSV ----
    df_c = pd.read_csv(csv_path)
    if xcol_csv not in df_c.columns or ycol_csv not in df_c.columns:
        raise KeyError("CSV columns not found")

    # ---- Convert to numpy + clean NaNs ----
    xm = df_m[xcol_minimpl].to_numpy()
    ym = df_m[ycol_minimpl].to_numpy()
    xc = df_c[xcol_csv].to_numpy()
    yc = df_c[ycol_csv].to_numpy()

    m_ok = np.isfinite(xm) & np.isfinite(ym)
    c_ok = np.isfinite(xc) & np.isfinite(yc)
    xm, ym = xm[m_ok], ym[m_ok]
    xc, yc = xc[c_ok], yc[c_ok]

    results = None

    # ---- Find peaks within peak_xlim ----
    if find_peak and peak_xlim is not None:
        lo, hi = peak_xlim
        m_mask = (xm >= lo) & (xm <= hi)
        c_mask = (xc >= lo) & (xc <= hi)

        if not np.any(m_mask):
            raise ValueError(f"No MiniMPL data in peak_xlim={peak_xlim}")
        if not np.any(c_mask):
            raise ValueError(f"No CSV data in peak_xlim={peak_xlim}")

        im = np.argmax(ym[m_mask])
        ic = np.argmax(yc[c_mask])

        peak_x_m = xm[m_mask][im]
        peak_y_m = ym[m_mask][im]
        peak_x_c = xc[c_mask][ic]
        peak_y_c = yc[c_mask][ic]

        results = {
            "ts": ts,
            "peak_xlim": peak_xlim,
            "minimpl_peak_x_m": float(peak_x_m),
            "minimpl_peak_y": float(peak_y_m),
            "csv_peak_x_m": float(peak_x_c),
            "csv_peak_y": float(peak_y_c),
            "peak_shift_m": float(peak_x_c - peak_x_m),  # csv - minimpl
        }

        print(f"[{ts}] Peak window: {lo}–{hi} m")
        print(f"  {label_minimpl} peak: x={peak_x_m:.1f} m, y={peak_y_m:.6g}")
        print(f"  {label_csv}     peak: x={peak_x_c:.1f} m, y={peak_y_c:.6g}")
        print(f"  Peak shift (csv - minimpl): {results['peak_shift_m']:.1f} m")

    # ---- Plot ----
    fig, ax = plt.subplots(figsize=figsize)

    # labels (optionally include peak info)
    lab_m = label_minimpl
    lab_c = label_csv
    if results and peak_in_legend:
        lab_m = f"{label_minimpl} (peak {results['minimpl_peak_x_m']:.0f} m)"
        lab_c = f"{label_csv} (peak {results['csv_peak_x_m']:.0f} m)"

    ax.plot(xm, ym, label=lab_m, linewidth=2)
    ax.plot(xc, yc, label=lab_c, linestyle="--")

    # draw peak markers
    if results:
        ax.axvline(results["minimpl_peak_x_m"], linestyle=":", linewidth=1.5)
        ax.axvline(results["csv_peak_x_m"], linestyle=":", linewidth=1.5)
        ax.scatter([results["minimpl_peak_x_m"]], [results["minimpl_peak_y"]], s=30)
        ax.scatter([results["csv_peak_x_m"]], [results["csv_peak_y"]], s=30)

        if annotate_peak:
            ax.text(
                results["minimpl_peak_x_m"], results["minimpl_peak_y"],
                f"{label_minimpl}\n{results['minimpl_peak_x_m']:.0f} m",
                fontsize=8, va="bottom", ha="left"
            )
            ax.text(
                results["csv_peak_x_m"], results["csv_peak_y"],
                f"{label_csv}\n{results['csv_peak_x_m']:.0f} m",
                fontsize=8, va="bottom", ha="right"
            )

        # peak summary box outside right (no fill)
        if peak_box:
            box_text = (
                f"Peak (within {results['peak_xlim'][0]}–{results['peak_xlim'][1]} m)\n"
                f"{label_minimpl}: {results['minimpl_peak_x_m']:.0f} m\n"
                f"{label_csv}: {results['csv_peak_x_m']:.0f} m\n"
                f"Shift (csv - minimpl): {results['peak_shift_m']:.0f} m"
            )
            ax.text(
                1.05, 0.98, box_text,
                transform=ax.transAxes,
                ha="left", va="top",
                fontsize=8,
                bbox=dict(
                    boxstyle="round,pad=0.3",
                    facecolor="none",
                    edgecolor="0.4",
                    linewidth=0.8
                )
            )

    ax.set_xlabel(xcol_minimpl)
    ax.set_ylabel(ycol_minimpl)
    ax.grid(True, alpha=0.3)
    ax.legend(loc=legend_loc)

    ax.set_xscale(xscale)
    ax.set_yscale(yscale)

    if xlim:
        ax.set_xlim(*xlim)
    if ylim:
        ax.set_ylim(*ylim)

    if title:
        ax.set_title(title)
    else:
        ax.set_title(f"{label_minimpl} vs {label_csv}\n{ts}")

    fig.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")

    if show:
        plt.show()

    plt.close(fig)
    return results


In [30]:
MINIMPL_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict")
PROTO_BASE = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV")

all_results = []

for day in DAYS:
    minimipl_csv = MINIMPL_DIR / f"MiniMPL_{day}.csv"
    if not minimipl_csv.exists():
        print(f"❌ Missing MiniMPL CSV: {minimipl_csv.name}")
        continue

    MiniMPL_day = pd.read_csv(minimipl_csv, parse_dates=["timestamp"])
    MiniMPL_day["timestamp"] = MiniMPL_day["timestamp"].dt.floor("min")

    # ✅ output folder ต่อวัน (ตรงนี้เท่านั้น!)
    out_dir = MINIMPL_DIR / day
    out_dir.mkdir(parents=True, exist_ok=True)

    # make the 30-min timestamps for this day
    date_obj = pd.to_datetime(day, format="%d-%m-%Y")
    date_str = date_obj.strftime("%Y-%m-%d")

    times_30min = pd.date_range(
        start=f"{date_str} 00:05",
        end=f"{date_str} 23:55",
        freq="30min"
    )

    print(f"\n📅 {day}: loop {len(times_30min)} timestamps")

    for ts in times_30min:
        ts = pd.Timestamp(ts).floor("min")
        hhmm = ts.strftime("%H%M")

        proto_csv = PROTO_BASE / f"{day}-{hhmm}" / f"{day}-{hhmm}_processed.csv"
        if not proto_csv.exists():
            print(f"⚠️ Missing Prototype CSV at {ts}")
            continue

        # ✅ output file ต่อ timestamp (ตรงนี้เท่านั้น!)
        out_png = out_dir / f"MiniMPL_vs_Prototype_{hhmm}.png"

        try:
            res = plot_minimpl_vs_csv(
                MiniMPL_day,
                ts=ts,
                csv_path=proto_csv,
                xcol_minimpl="range_m",
                ycol_minimpl="Normalize_copol_nrb",
                xcol_csv="range_m",
                ycol_csv="range2_norm",
                label_minimpl="MiniMPL",
                label_csv="Prototype",
                yscale="linear",
                xlim=(0, 5000),
                peak_xlim=(0, 5000),
                annotate_peak=False,
                peak_box=True,
                peak_in_legend=False,
                legend_loc="upper right",
                show=False,            # 👈 ไม่ popup
                save_path=out_png,     # 👈 save จริง
            )
            all_results.append(res)

        except ValueError as e:
            print(f"⚠️ Skip {ts}: {e}")



📅 20-01-2026: loop 48 timestamps
⚠️ Missing Prototype CSV at 2026-01-20 00:05:00
⚠️ Missing Prototype CSV at 2026-01-20 00:35:00
⚠️ Missing Prototype CSV at 2026-01-20 01:05:00
⚠️ Missing Prototype CSV at 2026-01-20 01:35:00
⚠️ Missing Prototype CSV at 2026-01-20 02:05:00
⚠️ Missing Prototype CSV at 2026-01-20 02:35:00
⚠️ Missing Prototype CSV at 2026-01-20 03:05:00
⚠️ Missing Prototype CSV at 2026-01-20 03:35:00
⚠️ Missing Prototype CSV at 2026-01-20 04:05:00
⚠️ Missing Prototype CSV at 2026-01-20 04:35:00
⚠️ Missing Prototype CSV at 2026-01-20 05:05:00
⚠️ Missing Prototype CSV at 2026-01-20 05:35:00
⚠️ Missing Prototype CSV at 2026-01-20 06:05:00
⚠️ Missing Prototype CSV at 2026-01-20 06:35:00
⚠️ Missing Prototype CSV at 2026-01-20 07:05:00
⚠️ Missing Prototype CSV at 2026-01-20 07:35:00
⚠️ Missing Prototype CSV at 2026-01-20 08:05:00
⚠️ Missing Prototype CSV at 2026-01-20 08:35:00
⚠️ Missing Prototype CSV at 2026-01-20 09:05:00
⚠️ Missing Prototype CSV at 2026-01-20 09:35:00
⚠️ Mis